# 章节练习

本章节涵盖 PD 分离架构原理、llm-datadist 接口和 omni-infer 框架，通过以下综合题目检验整体学习效果。如有错误，建议结合对应知识点进行复盘。

1. （判断题）PD 分离架构将 Prefill 和 Decode 部署到不同节点，主要目的是降低模型参数量。

2. （判断题）`LlmRole::kMix` 角色表示节点同时支持 Prefill 和 Decode 功能。

3. （判断题）双向链接需要 P 和 D 双方都配置 IP 列表并发起 link，而单向链接只需一方发起。

4. （判断题）逐层异步传输通过 `transfer_cache_async` 配合 `LayerSynchronizer` 实现，可以重叠计算与传输。

5. （单选题）Prefill 阶段的计算特征是？
    A. 访存密集型
    B. 计算密集型
    C. IO 密集型
    D. 网络密集型

6. （单选题）`CacheDesc` 中的 `num_tensors` 通常设为多少？
    A. num_layers
    B. 2 × num_layers
    C. batch_size
    D. num_heads

7. （单选题）xPyD 场景下，一个 Decode 节点从多个 Prefill 节点拉取 KV Cache，应使用哪种链接方式？
    A. 双向链接
    B. 单向链接
    C. 无法实现
    D. 广播链接

8. （单选题）`KvCacheExtParam` 中的 `tensor_num_per_layer` 默认值为多少？
    A. 1
    B. 2
    C. 4
    D. 8

9. （单选题）omni-infer 的 Global Proxy 中，`ngx_http_prefill_module` 生成的子请求发送到哪个路径？
    A. `/decode_internal`
    B. `/prefill_internal`
    C. `/kv_cache_transfer`
    D. `/pd_proxy`

10. （单选题）正确的 llm-datadist 资源释放顺序是？
    A. Finalize → Unregister → Unlink
    B. Unlink → Finalize → Unregister
    C. Unlink → Unregister → Finalize
    D. Unregister → Unlink → Finalize

11. （多选题）PD 分离架构的核心优势包括？
    A. 资源独立优化
    B. 批处理提升
    C. 弹性扩缩容
    D. 降低模型参数量

12. （多选题）llm-datadist 支持的 Cache 传输模式有？
    A. 连续 Cache 传输（PullKvCache/PushKvCache）
    B. 分块 Cache 传输（PullKvBlocks/PushKvBlocks）
    C. 逐层异步传输（transfer_cache_async）
    D. 本地拷贝（CopyKvCache/CopyKvBlocks）

13. （多选题）omni-infer 支持的负载均衡策略有？
    A. prefill_score_balance
    B. weighted_least_active
    C. pd_score_balance
    D. round_robin

14. （多选题）以下哪些是 llm-datadist 的状态码？
    A. LLM_SUCCESS
    B. LLM_KV_CACHE_NOT_EXIST
    C. LLM_NOT_YET_LINK
    D. LLM_LINK_FAILED

15. （多选题）关于 HIXL 在 PD 分离中的传输后端，以下说法正确的是？
    A. 可通过 `llm.TransferBackend` 配置
    B. 支持 HIXL 作为传输后端
    C. 传输后端切换不影响上层 API
    D. 传输后端决定了底层使用的物理链路

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/05.05_answer.txt

## 综合编程实践

以下为 3 道编程实践题（简单 / 中等 / 困难各一道），请按照要求补全代码中的 TODO 后运行验证。

| 任务 | 难度 | 环境要求 | 说明 |
| --- | --- | --- | --- |
| 任务一 | 简单 | 纯 Python | KV Cache 传输可行性分析 |
| 任务二 | 中等 | 纯 Python | llm-datadist API 调用流程模拟与验证 |
| 任务三 | 困难 | 昇腾 NPU + llm-datadist | Push 模式 KV Cache 传输实现 |


In [ ]:
# 创建练习目录
!mkdir -p src/05.05_chapter_practice


In [ ]:
%%writefile src/05.05_chapter_practice/kv_cache_transfer_analysis.py
# 综合编程实践·任务一（待填写）：KV Cache 传输可行性分析
# 纯 Python 实现，无需昇腾环境
# 补全 TODO，运行：
#   python src/05.05_chapter_practice/kv_cache_transfer_analysis.py

# 模型配置
models = [
    {"name": "LLaMA-7B",  "num_layers": 32, "num_heads": 32, "head_dim": 128, "dtype_size": 2},
    {"name": "LLaMA-65B", "num_layers": 80, "num_heads": 64, "head_dim": 128, "dtype_size": 2},
    {"name": "Qwen-72B",  "num_layers": 80, "num_heads": 64, "head_dim": 128, "dtype_size": 2},
]

# 链路配置
links = [
    {"name": "HCCS",      "bandwidth_gbps": 56},
    {"name": "RDMA",      "bandwidth_gbps": 100},
    {"name": "FabricMem", "bandwidth_gbps": 119},
]

batch_size = 8
seq_len = 2048
tpot_ms = 20  # 每 token 解码延迟（ms），用于判断实时性


def calc_kv_cache_size_gb(num_layers, batch_size, seq_len, num_heads, head_dim, dtype_size):
    """计算 KV Cache 大小（GB）

    公式：num_layers × 2 (K+V) × batch_size × seq_len × num_heads × head_dim × dtype_size
    """
    # TODO-1: 补全计算公式，返回 GB 单位的 KV Cache 大小
    pass


def calc_transfer_time_sec(size_gb, bandwidth_gbps):
    """计算传输时间（秒）

    Args:
        size_gb: 数据大小（GB）
        bandwidth_gbps: 链路带宽（GB/s）
    """
    # TODO-2: 补全传输时间计算
    pass


def check_feasibility(transfer_time_sec, tpot_ms, seq_len):
    """判断 PD 分离实时传输是否可行

    判断条件：KV Cache 传输时间 < Decode 阶段生成首个 token 的时间（TPOT × seq_len / 1000 秒）
    若传输时间小于该值，则计算与传输可重叠，实时分离可行。

    Args:
        transfer_time_sec: 传输时间（秒）
        tpot_ms: 每 token 解码延迟（ms）
        seq_len: 序列长度
    Returns:
        (feasible: bool, threshold_sec: float)
    """
    # TODO-3: 补全可行性判断
    pass


# ===== 以下为测试逻辑，补全 TODO 后无需修改 =====
if __name__ == "__main__":
    print("=" * 70)
    print("KV Cache 传输可行性分析")
    print("=" * 70)

    for model in models:
        cache_size = calc_kv_cache_size_gb(
            model["num_layers"], batch_size, seq_len,
            model["num_heads"], model["head_dim"], model["dtype_size"]
        )
        print(f"\n模型: {model['name']}")
        print(f"  配置: layers={model['num_layers']}, heads={model['num_heads']}, "
              f"head_dim={model['head_dim']}, batch={batch_size}, seq_len={seq_len}")
        print(f"  KV Cache 大小: {cache_size:.2f} GB")

        for link in links:
            transfer_time = calc_transfer_time_sec(cache_size, link["bandwidth_gbps"])
            feasible, threshold = check_feasibility(transfer_time, tpot_ms, seq_len)
            status = "✅ 可行" if feasible else "❌ 不可行"
            print(f"    {link['name']:12s} | 带宽 {link['bandwidth_gbps']:3d} GB/s | "
                  f"传输时间 {transfer_time:.4f}s | 阈值 {threshold:.4f}s | {status}")


In [ ]:
# 任务一：补全 TODO 后运行
!python src/05.05_chapter_practice/kv_cache_transfer_analysis.py


In [ ]:
%%writefile src/05.05_chapter_practice/datadist_flow_simulator.py
# 综合编程实践·任务二（待填写）：llm-datadist API 调用流程模拟与验证
# 纯 Python 实现，无需昇腾环境，使用 Mock 对象模拟 API 调用
# 补全 TODO，运行：
#   python src/05.05_chapter_practice/datadist_flow_simulator.py

from enum import Enum, auto


class LLMRole(Enum):
    PROMPT = auto()
    DECODER = auto()


class MockLLMDataDist:
    """模拟 llm-datadist"""
    def __init__(self):
        self.initialized = False
        self.linked = False
        self.finalized = False

    def init(self, options=""):
        self.initialized = True
        print("  [Mock] init() called")

    def link_clusters(self, clusters, timeout=5000):
        self.linked = True
        print(f"  [Mock] link_clusters({clusters}) called")

    def unlink_clusters(self, clusters, timeout=5000, force=True):
        self.linked = False
        print(f"  [Mock] unlink_clusters({clusters}) called")

    def finalize(self):
        self.finalized = True
        print("  [Mock] finalize() called")


class MockCacheManager:
    """模拟 CacheManager"""
    def __init__(self):
        self.registered = {}
        self.allocated = {}

    def register_blocks_cache(self, cache_key, mem_ptr, size):
        self.registered[cache_key] = (mem_ptr, size)
        print(f"  [Mock] register_blocks_cache({cache_key}) called")

    def allocate_cache(self, cache_desc):
        cache_id = f"cache_{len(self.allocated)}"
        self.allocated[cache_id] = cache_desc
        print(f"  [Mock] allocate_cache() -> {cache_id}")
        return cache_id

    def unregister_cache(self, cache_key):
        if cache_key in self.registered:
            del self.registered[cache_key]
            print(f"  [Mock] unregister_cache({cache_key}) called")

    def deallocate_cache(self, cache_id):
        if cache_id in self.allocated:
            del self.allocated[cache_id]
            print(f"  [Mock] deallocate_cache({cache_id}) called")


class FlowValidator:
    """验证 API 调用流程是否正确"""
    def __init__(self):
        self.call_sequence = []

    def record(self, step_name):
        self.call_sequence.append(step_name)

    def validate_prefill_push_flow(self):
        """验证 Prefill 端 Push 流程

        正确顺序：init → link → register → push → unlink → unregister → finalize
        """
        expected = ["init", "link", "register", "push", "unlink", "unregister", "finalize"]
        errors = []
        for i, expected_step in enumerate(expected):
            actual = self.call_sequence[i] if i < len(self.call_sequence) else "缺失"
            if actual != expected_step:
                errors.append(f"  步骤 {i+1}: 期望 '{expected_step}'，实际 '{actual}'")
        return errors


def simulate_prefill_push_flow():
    """模拟 Prefill 端 Push KV Cache 的完整流程"""
    validator = FlowValidator()
    datadist = MockLLMDataDist()
    cache_manager = MockCacheManager()
    decode_cluster = 1

    print("\n=== Prefill 端 Push 流程模拟 ===")

    # TODO-1: 初始化 LLMDataDist（角色为 PROMPT）
    # 提示：调用 datadist.init()，并调用 validator.record("init")
    pass

    # TODO-2: 建立集群链接
    # 提示：调用 datadist.link_clusters([decode_cluster])，并调用 validator.record("link")
    pass

    # TODO-3: 注册 KV Cache 内存
    # 提示：调用 cache_manager.register_blocks_cache("kv_cache_key", 0, 1073741824)
    #       并调用 validator.record("register")
    pass

    # 模拟 Push 传输（已有实现，无需修改）
    validator.record("push")
    print("  [Mock] push_blocks() called — KV Cache 已推送到 Decode 节点")

    # TODO-4: 资源释放（注意顺序！）
    # 正确顺序：unlink → unregister → deallocate → finalize
    # 提示：依次调用并记录：
    #   datadist.unlink_clusters([decode_cluster])  → validator.record("unlink")
    #   cache_manager.unregister_cache("kv_cache_key") → validator.record("unregister")
    #   cache_manager.deallocate_cache(cache_id)       → validator.record("deallocate")  ← 注意：deallocate 不在验证序列中，但应在 unregister 后调用
    #   datadist.finalize()                           → validator.record("finalize")
    pass

    # 验证流程
    errors = validator.validate_prefill_push_flow()
    if errors:
        print("\n❌ 流程验证失败：")
        for e in errors:
            print(e)
    else:
        print("\n✅ 流程验证通过！API 调用顺序正确。")


if __name__ == "__main__":
    simulate_prefill_push_flow()


In [ ]:
# 任务二：补全 TODO 后运行
!python src/05.05_chapter_practice/datadist_flow_simulator.py


### 任务三（困难）：Push 模式 KV Cache 传输实现

> **环境要求：** 昇腾 NPU + llm-datadist 环境

本任务要求实现 **Push 模式** 的 KV Cache 传输。教程 05.03 节展示了 Pull 模式（Decode 端主动拉取），本任务要求实现 Push 模式（Prefill 端主动推送）。

**Push 与 Pull 的关键区别：**

| 维度 | Pull 模式 | Push 模式 |
| --- | --- | --- |
| 发起方 | Decode 端调用 `pull_cache` | Prefill 端调用 `push_cache` |
| 链接方式 | 双向链接 | 单向链接（P→D） |
| Cache 角色 | Decode 端分配并 Pull | Prefill 端注册并 Push |

补全 `push_prefill.py` 和 `push_decode.py` 中的 TODO 后运行验证。


In [ ]:
%%writefile src/05.05_chapter_practice/push_prefill.py
# 综合编程实践·任务三（待填写）：Push 模式 — Prefill 端实现
# 需昇腾 NPU + llm-datadist 环境
# 补全 TODO 后，与 push_decode.py 一起运行

from llm_datadist import (
    LLMDataDist, LLMRole, LLMConfig, CacheDesc,
    Placement, DataType, BlocksCacheKey, RegisterMemStatus,
    LLMStatusCode
)
import torch
import torch_npu
import time

PREFILL_CLUSTER_ID = 0
num_layers = 2
batch_size = 1
num_heads = 8
seq_len = 128
head_dim = 128

# TODO-1: 初始化 LLMDataDist（Prefill 角色）
# 提示：创建 LLMDataDist(LLMRole.PROMPT, PREFILL_CLUSTER_ID)
#       配置 LLMConfig（device_id=0, enable_cache_manager=True）
#       调用 init(options)
pass

# TODO-2: 注册 KV Cache 内存
# 提示：分配 device 内存（torch_npu），调用 register_blocks_cache 注册
#       cache_key 使用 BlocksCacheKey(0, "push_cache_key")
pass

# TODO-3: 建立单向链接到 Decode 节点
# 提示：调用 link_clusters([1], timeout=5000)
#       注意：Push 模式使用单向链接
pass

# TODO-4: 执行 Push 传输
# 提示：等待链接建立后，调用 push_cache 或 push_blocks 将 KV Cache 推送到 Decode
#       使用 cache_manager.allocate_cache 分配 cache 后执行 push
pass

# TODO-5: 资源释放（正确顺序！）
# 提示：unlink_clusters → unregister_cache → deallocate_cache → finalize
pass

print("Prefill push completed!")


In [ ]:
%%writefile src/05.05_chapter_practice/push_decode.py
# 综合编程实践·任务三（待填写）：Push 模式 — Decode 端实现
# 需昇腾 NPU + llm-datadist 环境
# 补全 TODO 后，与 push_prefill.py 一起运行

from llm_datadist import (
    LLMDataDist, LLMRole, LLMConfig, CacheDesc,
    Placement, DataType, BlocksCacheKey, RegisterMemStatus,
    LLMStatusCode
)
import torch
import torch_npu
import time

DECODE_CLUSTER_ID = 1
PREFILL_CLUSTER_ID = 0
num_layers = 2
batch_size = 1
num_heads = 8
seq_len = 128
head_dim = 128

# TODO-1: 初始化 LLMDataDist（Decode 角色）
# 提示：创建 LLMDataDist(LLMRole.DECODER, DECODE_CLUSTER_ID)
#       配置 LLMConfig（device_id=1, enable_cache_manager=True, transfer_backend="hixl"）
pass

# TODO-2: 注册 KV Cache 内存（用于接收 Push 数据）
# 提示：分配 device 内存，注册 cache_key 与 Prefill 端一致
pass

# TODO-3: 接收 Push 数据
# 提示：Push 模式下 Decode 端被动接收，需确保 cache 已注册并等待传输完成
#       可通过轮询 cache 状态或使用同步机制等待
pass

# TODO-4: 资源释放（正确顺序！）
pass

print("Decode received push data!")


In [ ]:
%%writefile src/05.05_chapter_practice/run_push.sh
#!/bin/bash
set -e

python3 push_prefill.py &
prefill_pid=$!

python3 push_decode.py &
decode_pid=$!

wait "${prefill_pid}"
wait "${decode_pid}"

echo "Push transfer run success!"


In [ ]:
# 任务三：需昇腾 NPU + llm-datadist 环境
# 补全 push_prefill.py 和 push_decode.py 中的 TODO 后，取消下面一行的注释运行：
# !cd src/05.05_chapter_practice && bash run_push.sh
print("请在昇腾环境中补全 TODO 后，取消上方注释运行 push 传输")


## 参考答案

执行以下代码查看各任务参考答案。


In [ ]:
# 任务一参考答案
!cat ./answer/05.05_practice_answer_task1.py


In [ ]:
# 任务二参考答案
!cat ./answer/05.05_practice_answer_task2.py


In [ ]:
# 任务三参考答案（需昇腾环境运行）
!cat ./answer/05.05_practice_answer_task3.py
